# QFun-KAN Training Demo

This notebook trains a small 1D QFun-KAN prototype and reuses the same modules as `train.py`.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import pennylane.numpy as pnp

from qfun_layers.qkan_block import QKANBlock
from train import make_dataset, target_function, train_model
from qfun_layers.signed_encoding import mode_a_signed_encode, reconstruct_mode_a_signed

In [ ]:
x_train, y_train = make_dataset(n_samples=128, seed=1)
model = QKANBlock(input_dim=1, num_functions=3, n_qubits=5, mode='mode_a')
x_train.shape, y_train.shape

In [ ]:
model, x_train, y_train, losses = train_model(num_functions=3, n_qubits=5, mode='mode_a', n_samples=128, steps=150, lr=0.05)
losses[0], losses[-1]

In [ ]:
plt.figure(figsize=(6,3))
plt.plot(losses)
plt.title('Training loss')
plt.xlabel('Step')
plt.ylabel('MSE')
plt.show()

In [ ]:
x_line = np.linspace(-1,1,256).reshape(-1,1)
y_true = target_function(x_line[:,0])
y_pred = np.asarray(model.forward_batch(pnp.array(x_line)))

plt.figure(figsize=(6,4))
plt.scatter(np.asarray(x_train[:,0]), np.asarray(y_train), s=10, alpha=0.4, label='train')
plt.plot(x_line[:,0], y_true, label='target', lw=2)
plt.plot(x_line[:,0], y_pred, label='qkan', lw=2)
plt.legend()
plt.show()

In [ ]:
fig, axes = plt.subplots(model.num_functions, 1, figsize=(7, 2.3*model.num_functions), sharex=True)
for m in range(model.num_functions):
    axes[m].plot(np.asarray(model.x_grid), np.asarray(model.get_profile(m)))
    axes[m].set_title(f'Learned profile m={m}')
plt.tight_layout()
plt.show()

In [ ]:
m = 0
grid_vals = np.asarray(model.grid_values[m])
alpha, sign_bits = mode_a_signed_encode(grid_vals)
q_hat = reconstruct_mode_a_signed(alpha, sign_bits)

plt.figure(figsize=(6,3))
plt.plot(np.asarray(model.x_grid), np.abs(alpha)**2, label='|alpha|^2')
plt.plot(np.asarray(model.x_grid), q_hat, label='q_hat = p+ - p-')
plt.legend()
plt.title('Encoded amplitudes and reconstructed signed profile')
plt.show()